# Labeling the Dataset using the Teacher

What we do:
1) Run the teacher model on the dataset of inputs
2) Analyse the teacher dataset and label it

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [2]:
from core.types import *
from core.utils.huggingface_client import HuggingFaceClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from sklearn.cluster import DBSCAN
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from openai.types.responses import Response as OpenAIResponse

import os
import json
import numpy as np
import pandas as pd

In [18]:
inputs = LLMCommandingInput.load_inputs(
    path=Path("data/inputs/inputs.json"),
    gstype=DoomGameState
)

inputs_lookup = {inp.id: inp for inp in inputs}

In [21]:
# For now, only extract inputs to label
selected_inputs = [
    inp
    for inp in inputs
    if inp.selected_for_labelling
]

print(f"Selected inputs: {len(selected_inputs)}/{len(inputs)}")

Selected inputs: 50/2872


In [26]:
teacher_client = OpenAIClient[LLMCommandingInput, LLMCommandingOutput](
    model="gpt-5.1",
    mode=ProcessingMode.SEQUENTIAL, # Should be SEQUENTIAL to measure latency
    max_output_tokens=800,
    temperature=0.0,
    reasoning_effort='none',
    working_dir=Path('data/openai'),
)

In [27]:
# Prepare Prompt
base_prompt = Path("prompts/command-execution-template-level-3.md").read_text(encoding="utf-8")
doom_root = Path("prompts/ultimate_doom")

full_prompt = base_prompt
for md_file in doom_root.rglob("*.md"):
    rel = md_file.relative_to(doom_root).with_suffix("")
    tag = "_".join(part.upper() for part in rel.parts)
    value = md_file.read_text(encoding="utf-8")
    full_prompt = full_prompt.replace(f"<{tag}>", value)

# No tool call needed at level 3 - we use DSL

In [28]:
def format_input(inp: LLMCommandingInput) -> str:
    game_state = inp.game_state.state.to_prompt_ready()
    command = inp.user_command.command.command
    return f"Game State:\n{game_state}\n\nUser Command:\n{command}"


def parse_output(response: OpenAIResponse, input_id: str, latency: float) -> LLMCommandingOutput:
    return LLMCommandingOutput(
        input_id=input_id,
        actions=response.output_text,
        reason=None,
        latency=latency,
    )


def get_id(gse: LLMCommandingInput, idx: int) -> str:
    return gse.id

In [29]:
print(full_prompt)
print(format_input(inputs[3]))

# Your Role

You are a gaming assistant for the game The Ultimate Doom. Your goal is to provide direct assistance to a human player by responding to their commands with properly chosen game actions, in the **ACTION** Domain Specific Language (DSL).

TASK INPUTS:
- The current game state.
- A user command, expressed by the player you are assisting.

TASK OUTPUT:
- A syntactically correct program written in the ACTION DSL, translating the user command into instructions for the game.

# General Instructions

- Use the provided game state only if it is relevant to the user command.
- If the command cannot be fulfilled using the available actions, output a single FAIL instruction with a brief reason.
- Otherwise, output a valid ACTION DSL program that fulfills the user command.
- Use the minimum number of actions and the shortest durations necessary.
- Output ONLY valid ACTION DSL. Do not output explanations or natural language.


# The Game The Ultimate Doom

The Ultimate Doom is a first-p

In [30]:
outputs = teacher_client.process(
    dataset=selected_inputs, # TO CHANGE
    system_prompt=full_prompt,
    tools = [], # No tools at level 3
    format_input=format_input,
    parse_output=parse_output,
    get_id=get_id,
    # batch_size=200,
)

Processing mode: Sequential
Model: gpt-5.1
Processing 50 items sequentially
[1/50] Processing item with id=state-233-p2-uc0...
[2/50] Processing item with id=state-6603-p1-uc0...
[3/50] Processing item with id=state-7768-p3-uc1...
[4/50] Processing item with id=state-7825-p3-uc1...
[5/50] Processing item with id=state-8097-p0-uc0...
[6/50] Processing item with id=state-8097-p2-uc1...
[7/50] Processing item with id=state-11021-p0-uc1...
[8/50] Processing item with id=state-11066-p1-uc0...
[9/50] Processing item with id=state-11798-p1-uc1...
[10/50] Processing item with id=state-18060-p0-uc0...
[11/50] Processing item with id=state-18189-p1-uc2...
[12/50] Processing item with id=state-18456-p1-uc2...
[13/50] Processing item with id=state-21621-p0-uc0...
[14/50] Processing item with id=state-21621-p1-uc2...
[15/50] Processing item with id=state-21789-p1-uc0...
[16/50] Processing item with id=state-21789-p3-uc1...
[17/50] Processing item with id=state-21827-p1-uc1...
[18/50] Processing ite

In [31]:
# Clustering was already made earlier, so inputs are already partitioned.
# Now, considering this is just a test of the Teacher's quality (to save time):
# - Having retrieved the LLMCommandingOutputs, I can just prepare the csv
# - This time, every row in the csv should be set to be evaluated.
# For the full run: check if selected. (MAKE SURE TO CHANGE FILE NAME SO THAT I DO NOT HAVE TO REVALUATE IF THEY ALREADY CORRECT)

rows = []
for idx, output in enumerate(outputs):
    inp = inputs_lookup[output.input_id]

    row = LLMCommandingLabelledDataPoint(
        input_id=inp.id,
        game_state=inp.game_state.state.to_prompt_ready(),
        command=inp.user_command.command.command,
        command_intent=inp.user_command.command.intent,
        command_explicitness=inp.user_command.command.explicitness,
        command_atomicity=float(inp.user_command.command.atomicity),
        command_contextuality=float(inp.user_command.command.contextuality),
        game_actions=output.actions.__str__(),
        latency=output.latency,
        reason_if_failed=output.reason,
        cluster_id=inp.cluster_id,
        selected_for_labelling=inp.selected_for_labelling,
    )

    rows.append(asdict(row))

df = pd.DataFrame(rows)

In [32]:
output_path = Path("data/outputs/selected-data-no-reasoning.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)